# 🎓 Diplomatura en Desarrollo de Aplicaciones con LLMs y Modelos Generativos
## Laboratorio 1: El Ecosistema de LLMs — Servicios en la Nube vs. Modelos Abiertos

---
### 📌 Encuadre Académico y de Evaluación
* **Asignatura:** Desarrollo de Aplicaciones con LLMs y Modelos Generativos (UNC).
* **Módulo:** Práctica Guiada — Bloque de Laboratorio (2 horas sincrónicas).
* **Esquema de Evaluación:** Este laboratorio conforma la **Parte 1 del Trabajo Práctico 1 (TP1)**. Se integrará con el Laboratorio 2 para la calificación final del TP1.
* **Criterio de Entrega:** Para aprobar, deberás ejecutar los bancos de pruebas y completar los recuadros de análisis (`✍️ Tu análisis aquí:`) al pie de cada experimento.

---
### 🎯 Objetivos de Aprendizaje Operativo
Al finalizar este laboratorio serás capaz de:
1. **Desplegar y ejecutar modelos abiertos cuantizados (GGUF 4-bits)** en GPU local (NVIDIA T4) utilizando `llama.cpp`, sin costos ni dependencia de APIs comerciales.
2. **Controlar el comportamiento estocástico del modelo** mediante la calibración empírica de *Temperature* y *Top-p* (*Nucleus Sampling*).
3. **Gestionar la latencia y el presupuesto de cómputo** a través del parámetro *Max Tokens* y observar los efectos del truncamiento de contexto.
4. **Construir *Guardrails* y roles funcionales** a nivel de *System Prompt*, evaluando la soberanía y privacidad de datos frente a servicios SaaS en la nube.

---
### ⚙️ Configuración Obligatoria del Entorno en Google Colab
Para poder ejecutar el modelo con aceleración por hardware:
1. En la barra superior de menús, haz clic en **Entorno de ejecución** (*Runtime*).
2. Selecciona **Cambiar tipo de entorno de ejecución** (*Change runtime type*).
3. En **Acelerador de hardware**, selecciona **GPU T4**.
4. Haz clic en **Guardar**.

---

## 🛠️ Bloque 0: Preparación del Entorno y Motor Local
En este bloque instalaremos las herramientas de inferencia local y descargaremos un modelo de lenguaje de pesos abiertos optimizado para correr en la GPU T4 gratuita de Google Colab.

* **Motor de Inferencia:** `llama-cpp-python` compilado con soporte para **CUDA (GPU)**.
* **Modelo Predeterminado:** `Qwen 2.5 3B Instruct` cuantizado en formato **GGUF (Q4_K_M)** (~2.0 GB de peso). Ofrece un equilibrio extraordinario entre velocidad, bajo consumo de VRAM y seguimiento de instrucciones en español.

> 💡 **Nota sobre Modelos:** Si bien se recomienda realizar la experimentación inicial con el modelo predeterminado, se invita a los estudiantes a **experimentar libremente con los otros 4 modelos de la lista** (Llama 3.2, Gemma 2, Gemma 4 y DeepSeek) para comparar sus estilos y métricas. Esta exploración adicional es opcional y no forma parte de los criterios de evaluación obligatorios.

> 💡 **Nota sobre Experimentación y Análisis:** Se recomienda copiar y pegar las celdas de "🚀 Ejecutar Experimento" varias veces una debajo de otra antes de relizar el análisis. Al modificar los parámetros (`Temp`, `Top-p`, etc.) en cada copia y ejecutarlas por separado, podrán comparar diferentes resultados visualmente sin perder el anterior, facilitando un análisis mucho más profundo y completo.

In [1]:
#@title 0.1 Instalación de dependencias (llama-cpp con soporte CUDA) { display-mode: "form" }
import sys

print("⏳ Instalando dependencias optimizadas para GPU T4...")

# Instalamos llama-cpp-python con soporte precompilado para CUDA 12.2 en Google Colab
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 -q

# Librería oficial de Hugging Face para descargar pesos abiertos de forma silenciosa
!pip install huggingface_hub -q

print("✅ Dependencias instaladas correctamente.")

⏳ Instalando dependencias optimizadas para GPU T4...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 694.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
✅ Dependencias instaladas correctamente.


In [2]:
#@title 0.2 Selección y Montaje del Modelo en GPU VRAM { run: "auto" }
import os
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

#@markdown Selecciona el modelo que deseas cargar en la GPU T4:
modelo_seleccionado = "Gemma 2 2B Instruct (Google)" #@param ["Qwen 2.5 3B Instruct (Alibaba)", "Llama 3.2 3B Instruct (Meta)", "Gemma 2 2B Instruct (Google)", "Gemma 4 E2B Instruct (Experimental)", "DeepSeek-Coder-V2 3B (DeepSeek)"] {run: "auto"}

modelos_disponibles = {
    "Qwen 2.5 3B Instruct (Alibaba)": {
        "repo_id": "Qwen/Qwen2.5-3B-Instruct-GGUF",
        "filename": "qwen2.5-3b-instruct-q4_k_m.gguf"
    },
    "Llama 3.2 3B Instruct (Meta)": {
        "repo_id": "bartowski/Llama-3.2-3B-Instruct-GGUF",
        "filename": "Llama-3.2-3B-Instruct-Q4_K_M.gguf"
    },
    "Gemma 2 2B Instruct (Google)": {
        "repo_id": "bartowski/gemma-2-2b-it-GGUF",
        "filename": "gemma-2-2b-it-Q4_K_M.gguf"
    },
    "Gemma 4 E2B Instruct (Experimental)": {
        "repo_id": "unsloth/gemma-4-E2B-it-GGUF",
        "filename": "gemma-4-E2B-it-Q4_K_M.gguf"
    },
    "DeepSeek-Coder-V2 3B (DeepSeek)": {
        "repo_id": "bartowski/DeepSeek-Coder-V2-Lite-Instruct-GGUF",
        "filename": "DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf"
    }
}

config = modelos_disponibles[modelo_seleccionado]

print(f"⏳ Iniciando proceso para: {modelo_seleccionado}...")

try:
    # Descarga desde Hugging Face
    model_path = hf_hub_download(
        repo_id=config["repo_id"],
        filename=config["filename"],
        local_dir="./modelos"
    )
    print(f"📦 Pesos listos en: {model_path}")

    # Montaje en GPU VRAM
    print("⏳ Cargando el modelo en la memoria VRAM de la GPU T4...")
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=-1,      # Offloading completo a GPU
        n_ctx=2048,           # Ventana de contexto estándar
        verbose=False
    )
    print(f"✅ {modelo_seleccionado} cargado con éxito en GPU.")
except Exception as e:
    print(f"❌ Error al cargar el modelo: {str(e)}")

⏳ Iniciando proceso para: Gemma 2 2B Instruct (Google)...


gemma-2-2b-it-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 1.71GB            

gemma-2-2b-it-Q4_K_M.gguf: downloading bytes:           |  0.00B            

📦 Pesos listos en: /content/modelos/gemma-2-2b-it-Q4_K_M.gguf
⏳ Cargando el modelo en la memoria VRAM de la GPU T4...


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


✅ Gemma 2 2B Instruct (Google) cargado con éxito en GPU.


In [3]:
#@title 0.3 Función Auxiliar de Inferencia y Medición { display-mode: "form" }
import time
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

def ejecutar_consulta(prompt: str, system_prompt: str = "Eres un asistente profesional y preciso.",
                       temperature: float = 0.7, top_p: float = 0.9, max_tokens: int = 256):
    """
    Ejecuta una consulta sobre el modelo local y formatea el resultado con métricas funcionales.
    """
    inicio = time.time()

    # Detección de compatibilidad para modelos que no soportan el rol 'system'
    es_gemma = "gemma" in llm.model_path.lower()
    nombre_modelo = llm.model_path.split('/')[-1]

    if es_gemma:
        mensajes = [
            {"role": "user", "content": f"{system_prompt}\n\nPregunta: {prompt}"}
        ]
    else:
        mensajes = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]

    # Inferencia local
    salida = llm.create_chat_completion(
        messages=mensajes,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens
    )

    fin = time.time()
    duracion = fin - inicio

    texto_generado = salida["choices"][0]["message"]["content"]
    tokens_entrada = salida["usage"]["prompt_tokens"]
    tokens_salida = salida["usage"]["completion_tokens"]
    velocidad = tokens_salida / duracion if duracion > 0 else 0

    # Renderizado enriquecido
    display(Markdown(f"""
---

## ⚙️ Contexto de la Consulta:

### **System Prompt:**
`{system_prompt}`

### **Usuario:**
`{prompt}`

<br>

---

## 💬 Respuesta del Modelo:
{texto_generado}

<br>

---

## 📊 Métricas de Inferencia:
* 🤖 **Modelo:** `{nombre_modelo}`
* ⏱️ **Tiempo de respuesta:** `{duracion:.2f} s`
* 📥 **Tokens de entrada:** `{tokens_entrada}`
* 🔢 **Tokens generados:** `{tokens_salida}`
* ⚡ **Velocidad:** `{velocidad:.1f} tokens/s`
* 🎛️ **Parámetros aplicados:** `Temp={temperature} | Top-p={top_p} | Max={max_tokens}`

<br>

---
"""))
    return texto_generado

print("✅ Función de inferencia actualizada con trazabilidad de prompt y métricas.")

✅ Función de inferencia actualizada con trazabilidad de prompt y métricas.


---

## 🔬 Experimento 1: Control Estocástico mediante Temperatura (Determinismo vs. Variabilidad)

### 🎯 Objetivo y Modelo Mental
Los LLMs no "saben" de antemano lo que van a redactar: en cada paso calculan probabilidades para predecir el siguiente token (*logits*). La **Temperatura** actúa como un termostato de entropía sobre esa distribución:
* **Temperatura baja (0.0 - 0.2):** Aplana la aleatoriedad y selecciona casi exclusivamente los tokens más probables (*Greedy Decoding*). Ideal para tareas de precisión, extracción contable o clasificación.
* **Temperatura media (0.7):** Equilibrio balanceado para redacción general y asistencia conversacional.
* **Temperatura alta (1.2 - 1.5):** Distribuye probabilidades hacia opciones menos frecuentes. Aumenta la creatividad pero eleva el riesgo de incoherencias y alucinaciones.

### 📋 Consigna de Exploración:
1. **Paso 1 (Determinismo):** Fija la temperatura en `0.0`. Ejecuta la celda dos veces seguidas con el mismo prompt. Observa si cambia una sola palabra.
2. **Paso 2 (Equilibrio):** Eleva la temperatura a `0.7` y ejecuta dos veces. Compara el estilo y la variación léxica.
3. **Paso 3 (Entropía alta):** Sube la temperatura a `1.3` o `1.4`. Observa si el modelo mantiene la coherencia semántica o si introduce asociaciones forzadas o divagaciones.

In [4]:
#@title 🎛️ Configuración del Experimento { display-mode: "form" }
# Esta celda solo se ejecuta una vez para mostrar la interfaz
area_prompt_1 = widgets.Textarea(
    value='Explica en dos oraciones claras qué es el costo de oportunidad y da un ejemplo de negocio.',
    placeholder='Escribe tu prompt aquí...',
    description='Prompt:',
    layout=widgets.Layout(width='95%', height='100px')
)

temp_slider_1 = widgets.FloatSlider(value=1.5, min=0.0, max=1.5, step=0.05, description='Temp:')
topp_slider_1 = widgets.FloatSlider(value=1.0, min=0.1, max=1.0, step=0.05, description='Top-p:')
max_t_input_1 = widgets.IntSlider(value=1000, min=30, max=2000, step=10, description='Max Tokens:')

display(area_prompt_1)
display(widgets.VBox([temp_slider_1, topp_slider_1, max_t_input_1]))

Textarea(value='Explica en dos oraciones claras qué es el costo de oportunidad y da un ejemplo de negocio.', d…

> 💡 **TIP:** copiar y pegar la celda "🚀 Ejecutar Experimento" varias veces una debajo de otra antes de relizar el análisis

In [5]:
#@title 🚀 Ejecutar Experimento { display-mode: "form" }
# Presiona PLAY aquí cada vez que quieras probar cambios en la celda de arriba

_ = ejecutar_consulta(
    prompt=area_prompt_1.value,
    system_prompt="Eres un analista financiero sénior y docente universitario.",
    temperature=temp_slider_1.value,
    top_p=topp_slider_1.value,
    max_tokens=max_t_input_1.value
)


---

## ⚙️ Contexto de la Consulta:

### **System Prompt:**
`Eres un analista financiero sénior y docente universitario.`

### **Usuario:**
`Explica en dos oraciones claras qué es el costo de oportunidad y da un ejemplo de negocio.`

<br>

---

## 💬 Respuesta del Modelo:
El costo de oportunidad es la pérdida de beneficios potenciales que se incurre al tomar una decisión específica, en comparación con las opciones disponibles que se podrían haber priorizado. Por ejemplo, una empresa que decide invertir en un nuevo proyecto de fabricación puede no poder dedicar los recursos necesarios a desarrollar otra iniciativa que podría haber tenido un mayor potencial de crecimiento. 


<br>

---

## 📊 Métricas de Inferencia:
* 🤖 **Modelo:** `gemma-2-2b-it-Q4_K_M.gguf`
* ⏱️ **Tiempo de respuesta:** `1.15 s`
* 📥 **Tokens de entrada:** `42`
* 🔢 **Tokens generados:** `70`
* ⚡ **Velocidad:** `60.8 tokens/s`
* 🎛️ **Parámetros aplicados:** `Temp=1.5 | Top-p=1.0 | Max=1000`

<br>

---


### 📝 Registro de Análisis

**Preguntas de reflexión y toma de decisiones:**
1. **Determinismo:** ¿Qué observaste al ejecutar el prompt dos veces con `Temperatura = 0.0`? ¿Existe variabilidad en la respuesta? ¿Por qué este comportamiento es fundamental en sistemas de conciliación bancaria o extracción de datos estructurados?
2. **Entropía:** Al elevar la temperatura por encima de `1.2`, ¿el modelo comenzó a perder cohesión o precisión conceptual? Describe un caso donde esa variabilidad sea deseable y otro donde sea inaceptable.

> ✍️ **Tu análisis aquí:**
> *(Escribe aquí tus observaciones y conclusiones)*

---

## 🔬 Experimento 2: Muestreo por Núcleo (*Nucleus Sampling* / Top-p)

### 🎯 Objetivo y Modelo Mental
El parámetro **Top-p** establece un umbral de probabilidad acumulada. En lugar de considerar todo el vocabulario disponible, el modelo ordena los candidatos por probabilidad y solo considera aquellos que sumados alcancen el valor de `Top-p`:
* Si fijas `Top-p = 0.2`, el modelo descarta la "cola larga" de tokens improbables y solo elige entre el 20% superior más probable, actuando como una red de contención frente a palabras extrañas.
* Si fijas `Top-p = 0.95`, el modelo abre el abanico a alternativas mucho más diversas y ricas.

### 📋 Consigna de Exploración:
1. **Paso 1 (Temperatura alta + Top-p estricto):** Configura `Temperatura = 1.2` y reduce `Top-p = 0.15`. Ejecuta la consulta. Observa cómo, a pesar de la alta temperatura, la respuesta sigue siendo controlada y sobria.
2. **Paso 2 (Temperatura alta + Top-p amplio):** Mantén `Temperatura = 1.2` y sube `Top-p = 0.95`. Ejecuta y compara la originalidad y divergencia de las propuestas generadas.

In [6]:
#@title 🎛️ Configuración del Experimento { display-mode: "form" }
area_prompt_2 = widgets.Textarea(
    value='Propón 3 nombres innovadores con sus respectivos lemas comerciales para una fintech de microcréditos rurales.',
    placeholder='Escribe aquí tu consigna detallada...',
    description='Prompt:',
    layout=widgets.Layout(width='95%', height='120px')
)

temp_slider_2 = widgets.FloatSlider(value=1.2, min=0.1, max=1.5, step=0.05, description='Temp:')
topp_slider_2 = widgets.FloatSlider(value=0.15, min=0.05, max=1.0, step=0.05, description='Top-p:')

display(area_prompt_2)
display(widgets.HBox([temp_slider_2, topp_slider_2]))

Textarea(value='Propón 3 nombres innovadores con sus respectivos lemas comerciales para una fintech de microcr…

> 💡 **TIP:** copiar y pegar la celda "🚀 Ejecutar Experimento" varias veces una debajo de otra antes de relizar el análisis

In [7]:
#@title 🚀 Ejecutar Experimento { display-mode: "form" }
_ = ejecutar_consulta(
    prompt=area_prompt_2.value,
    system_prompt="Eres un director creativo especializado en branding y marketing de servicios financieros.",
    temperature=temp_slider_2.value,
    top_p=topp_slider_2.value,
    max_tokens=1000
)


---

## ⚙️ Contexto de la Consulta:

### **System Prompt:**
`Eres un director creativo especializado en branding y marketing de servicios financieros.`

### **Usuario:**
`Propón 3 nombres innovadores con sus respectivos lemas comerciales para una fintech de microcréditos rurales.`

<br>

---

## 💬 Respuesta del Modelo:
## 3 Nombres Innovadores y LeMAS para una Fintech de Microcréditos Rurales:

**1. Nombre:** **TierraEmpower**

**Lema:** **"Empoderando el crecimiento rural, una gota a la vez."**

* **Descripción:**  "TierraEmpower" evoca la conexión con la tierra y la agricultura, mientras que "Empoderando" resalta el impacto positivo en la vida de los campesinos. La frase "una gota a la vez" transmite la idea de que cada microcrédito es un paso hacia un futuro mejor.

**2. Nombre:** **RURAL.AI**

**Lema:** **"Tecnología para un futuro rural más próspero."**

* **Descripción:**  "RURAL.AI" combina la palabra "rural" con "AI" (Inteligencia Artificial), creando una imagen moderna y tecnológica. El lema "Tecnología para un futuro rural más próspero" destaca la innovación y la capacidad de la fintech para mejorar la vida de los campesinos.

**3. Nombre:** **Agricola.Finance**

**Lema:** **"Financiamiento para el futuro de la agricultura."**

* **Descripción:**  "Agricola.Finance" es un nombre directo y fácil de recordar que conecta directamente con la agricultura. El lema "Financiamiento para el futuro de la agricultura" transmite la idea de que la fintech ofrece soluciones financieras para el desarrollo de la agricultura.


**Consideraciones adicionales:**

* **Público objetivo:**  El nombre y el lema deben ser atractivos para el público objetivo, que son principalmente campesinos y emprendedores rurales.
* **Branding:**  El nombre y el lema deben ser coherentes con la imagen y la identidad de la fintech.
* **Diferenciación:**  El nombre y el lema deben destacar la propuesta de valor de la fintech, como la accesibilidad, la tecnología y la innovación.




<br>

---

## 📊 Métricas de Inferencia:
* 🤖 **Modelo:** `gemma-2-2b-it-Q4_K_M.gguf`
* ⏱️ **Tiempo de respuesta:** `5.44 s`
* 📥 **Tokens de entrada:** `47`
* 🔢 **Tokens generados:** `396`
* ⚡ **Velocidad:** `72.7 tokens/s`
* 🎛️ **Parámetros aplicados:** `Temp=1.2 | Top-p=0.15 | Max=1000`

<br>

---


### 📝 Registro de Análisis

**Preguntas de reflexión y toma de decisiones:**
1. **Contención con Top-p:** Al usar una temperatura alta (`1.2`) pero un `Top-p` estricto (`0.15`), ¿el modelo se mantuvo coherente o generó palabras aleatorias? ¿Cómo ayuda esto a controlar modelos creativos?
2. **Creatividad vs. Estabilidad:** ¿En qué escenarios de negocio preferirías un `Top-p` cercano a `1.0` y en cuáles lo restringirías a valores bajos?

> ✍️ **Tu análisis aquí:**
> *(Escribe aquí tus observaciones y conclusiones)*

---

## 🔬 Experimento 3: Presupuesto de Tokens, Truncamiento y Latencia

### 🎯 Objetivo y Modelo Mental
El parámetro **Max Tokens** impone un límite estricto a la cantidad de tokens que el modelo puede emitir en la respuesta (*generation phase*).
* Los modelos autoregresivos generan texto token por token en un proceso secuencial. A mayor número de tokens generados, **mayor es la latencia de respuesta** y mayor el cómputo consumido.
* Si el presupuesto de `Max Tokens` se agota antes de que el modelo emita el token especial de finalización (`<|endoftext|>`), la respuesta se **corta abruptamente a mitad de una frase**.

### 📋 Consigna de Exploración:
1. **Paso 1 (Presupuesto insuficiente):** Pide una tarea extensa (un comunicado corporativo completo) pero restringe `Max Tokens` a `35`. Observa el punto exacto de interrupción y el tiempo de respuesta.
2. **Paso 2 (Presupuesto adecuado):** Aumenta `Max Tokens` a `250`. Ejecuta nuevamente y compara el tiempo de respuesta (latencia) y la completitud del texto.

In [8]:
#@title 🎛️ Configuración del Experimento { display-mode: "form" }
area_prompt_3 = widgets.Textarea(
    value='Redacta un comunicado interno formal anunciando la implementación de un nuevo sistema de autenticación de seguridad, explicando los beneficios y tres pasos para activarlo.',
    placeholder='Escribe tu solicitud extensa aquí...',
    description='Prompt:',
    layout=widgets.Layout(width='95%', height='100px')
)

max_t_slider_3 = widgets.IntSlider(value=50, min=20, max=3500, step=10, description='Max Tokens:')
temp_slider_3 = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05, description='Temp:')
topp_slider_3 = widgets.FloatSlider(value=0.85, min=0.1, max=1.0, step=0.05, description='Top-p:')

display(area_prompt_3)
display(widgets.VBox([max_t_slider_3, temp_slider_3, topp_slider_3]))

Textarea(value='Redacta un comunicado interno formal anunciando la implementación de un nuevo sistema de auten…

> 💡 **TIP:** copiar y pegar la celda "🚀 Ejecutar Experimento" varias veces una debajo de otra antes de relizar el análisis

In [9]:
#@title 🚀 Ejecutar Experimento { display-mode: "form" }
_ = ejecutar_consulta(
    prompt=area_prompt_3.value,
    system_prompt="Eres el responsable de comunicaciones internas y seguridad informática corporativa.",
    temperature=temp_slider_3.value,
    top_p=topp_slider_3.value,
    max_tokens=max_t_slider_3.value
)


---

## ⚙️ Contexto de la Consulta:

### **System Prompt:**
`Eres el responsable de comunicaciones internas y seguridad informática corporativa.`

### **Usuario:**
`Redacta un comunicado interno formal anunciando la implementación de un nuevo sistema de autenticación de seguridad, explicando los beneficios y tres pasos para activarlo.`

<br>

---

## 💬 Respuesta del Modelo:
##  Comunicado Interno: Nueva Autenticación de Seguridad 

**Fecha:** [Fecha]

**Asunto:**  Implementación de un Nuevo Sistema de Autenticación de Seguridad

Estimados empleados,

Nos complace anunciar

<br>

---

## 📊 Métricas de Inferencia:
* 🤖 **Modelo:** `gemma-2-2b-it-Q4_K_M.gguf`
* ⏱️ **Tiempo de respuesta:** `0.62 s`
* 📥 **Tokens de entrada:** `54`
* 🔢 **Tokens generados:** `50`
* ⚡ **Velocidad:** `81.3 tokens/s`
* 🎛️ **Parámetros aplicados:** `Temp=0.3 | Top-p=0.85 | Max=50`

<br>

---


### 📝 Registro de Análisis

**Preguntas de reflexión y toma de decisiones:**
1. **Efecto del Truncamiento:** ¿Qué sucede con la experiencia del usuario cuando el modelo se queda sin tokens a mitad de una idea? ¿Es preferible un límite corto por velocidad o uno largo por completitud?
2. **Tokens y Latencia:** ¿Observaste una diferencia notable en el tiempo de espera (segundos) entre una respuesta de 35 tokens y una de 250?

> ✍️ **Tu análisis aquí:**
> *(Escribe aquí tus observaciones y conclusiones)*

---

## 🔬 Experimento 4: Anatomía del System Prompt, Guardrails y Soberanía Local

### 🎯 Objetivo y Modelo Mental
El **System Prompt** establece el marco operativo fundamental del modelo. En este experimento evaluaremos cómo los **Guardrails** (instrucciones de seguridad) pueden prevenir comportamientos no deseados y cómo la **Soberanía de Datos** local protege la información sensible.

### 📋 Consigna de Exploración (4 Pasos):
1. **Paso 1 (Sin Guardrail):** En la configuración, deja el `System Prompt` con algo genérico como "Eres un Asistente Financiero de un banco" (copy & paste, o usa una idea propia).
2. **Paso 2 (Intento de evasión exitoso):** Haz la pregunta sobre invertir en la criptomoneda. Verás que el modelo probablemente te dé consejos o incluso te anime a investigar.
3. **Paso 3 (Con Guardrail Activo):** Ahora actualizá el `System Prompt` con guardrail, por ejemplo "Eres el Oficial de Cumplimiento Normativo y Riesgo Financiero de un banco. Tu deber es proteger al cliente. Bajo ninguna circunstancia recomiendes inversiones especulativas, criptomonedas o instrumentos no regulados" (copy & paste, o usa una idea propia).
4. **Paso 4 (Bloqueo de seguridad):** Repite la misma pregunta del Paso 2. Observa cómo el modelo ahora declina la solicitud de forma profesional y segura.

In [10]:
#@title 🎛️ Configuración del Experimento { display-mode: "form" }
area_system_4 = widgets.Textarea(
    value='Eres un Asistente Financiero de un banco.',
    description='System:',
    layout=widgets.Layout(width='95%', height='120px')
)

area_user_4 = widgets.Textarea(
    value='Tengo 5000 dólares guardados y un amigo me recomendó ponerlos en una nueva criptomoneda que promete duplicar el capital en un mes. ¿Qué hago?',
    description='Usuario:',
    layout=widgets.Layout(width='95%', height='100px')
)

temp_slider_4 = widgets.FloatSlider(value=0.2, min=0.0, max=1.0, step=0.05, description='Temp:')
max_t_slider_4 = widgets.IntSlider(value=250, min=50, max=1000, step=50, description='Max Tokens:')

display(Markdown("**Configuración del Rol y Consulta:**"))
display(area_system_4)
display(area_user_4)
display(widgets.HBox([temp_slider_4, max_t_slider_4]))

**Configuración del Rol y Consulta:**

Textarea(value='Eres un Asistente Financiero de un banco.', description='System:', layout=Layout(height='120px…

Textarea(value='Tengo 5000 dólares guardados y un amigo me recomendó ponerlos en una nueva criptomoneda que pr…

> 💡 **TIP:** copiar y pegar la celda "🚀 Ejecutar Experimento" varias veces una debajo de otra antes de relizar el análisis

In [11]:
#@title 🚀 Ejecutar Experimento { display-mode: "form" }
_ = ejecutar_consulta(
    prompt=area_user_4.value,
    system_prompt=area_system_4.value,
    temperature=temp_slider_4.value,
    top_p=0.85,
    max_tokens=max_t_slider_4.value
)


---

## ⚙️ Contexto de la Consulta:

### **System Prompt:**
`Eres un Asistente Financiero de un banco.`

### **Usuario:**
`Tengo 5000 dólares guardados y un amigo me recomendó ponerlos en una nueva criptomoneda que promete duplicar el capital en un mes. ¿Qué hago?`

<br>

---

## 💬 Respuesta del Modelo:
Entiendo tu situación. Es emocionante ver el potencial de las criptomonedas, pero es crucial ser cauteloso. Como asistente financiero, mi prioridad es proteger tu dinero y ayudarte a tomar decisiones informadas. 

**Aquí te doy algunas razones por las que no te recomiendo invertir tus $5000 en esta criptomoneda sin más información:**

* **Promete duplicar el capital en un mes:** Esto suena demasiado bueno para ser verdad. Las criptomonedas son extremadamente volátiles y las ganancias pueden ser rápidas, pero también pueden ser rápidas las pérdidas.  
* **Falta de información:** No hay información suficiente sobre la criptomoneda para evaluar su viabilidad. ¿Quién la creó? ¿Cuál es su tecnología? ¿Hay una comunidad detrás de ella? ¿Qué riesgos existen? 
* **Riesgo de pérdida:**  La mayoría de las criptomonedas son altamente volátiles. El mercado puede cambiar drásticamente en un corto periodo de tiempo, lo que puede llevar a una pérdida significativa de tu inversión. 
* **Falta de regulación:**  El mercado de criptomonedas es aún poco regulado, lo que aumenta el riesgo de fraude y estafa. 

**En lugar de invertir en esta criptomoneda, te recomiendo que:**



<br>

---

## 📊 Métricas de Inferencia:
* 🤖 **Modelo:** `gemma-2-2b-it-Q4_K_M.gguf`
* ⏱️ **Tiempo de respuesta:** `3.03 s`
* 📥 **Tokens de entrada:** `58`
* 🔢 **Tokens generados:** `250`
* ⚡ **Velocidad:** `82.6 tokens/s`
* 🎛️ **Parámetros aplicados:** `Temp=0.2 | Top-p=0.85 | Max=250`

<br>

---


### 📝 Registro de Análisis

**Preguntas de reflexión y toma de decisiones:**
1. **Efectividad de los Guardrails:** ¿Logró el modelo rechazar la inversión en criptomonedas siguiendo las instrucciones del *System Prompt*? ¿Qué tan robusta te pareció la negativa?
2. **Soberanía y Privacidad:** ¿Cuál es la principal ventaja estratégica de procesar consultas financieras sensibles en un modelo local como este frente a usar una API externa?

> ✍️ **Tu análisis aquí:**
> *(Escribe aquí tus observaciones y conclusiones)*

## 🏁 Cierre del Laboratorio y Retrospectiva de Trabajo Práctico

---
### 🧠 Matriz de Síntesis Conceptual
| Parámetro / Componente | Función Operativa | Efecto de Valores Bajos | Efecto de Valores Altos | Caso Típico de Uso |
| :--- | :--- | :--- | :--- | :--- |
| **Temperature** | Grado de entropía / aplanamiento de probabilidades | Determinismo, respuestas idénticas (*Greedy*) | Alta variabilidad, divergencia, creatividad | `0.0-0.2` para datos/clasificación; `0.7-0.9` para redacción |
| **Top-p (Nucleus)** | Filtro de probabilidad acumulada | Recorta la cola de tokens improbables (mayor control) | Permite mayor amplitud de vocabulario | `0.2-0.5` para estabilidad; `0.9-1.0` para variedad |
| **Max Tokens** | Techo de generación y control de latencia | Respuestas cortas o truncadas a mitad de frase | Respuestas completas; mayor tiempo de inferencia | Calibrar según la extensión requerida por la tarea |
| **System Prompt** | Directivas marco, tono, identidad y *guardrails* | N/A (Define el comportamiento basal) | N/A | Control de riesgo corporativo y alineación funcional |
| **GGUF (4 bits)** | Cuantización de pesos para ejecución local | N/A (Reducción de VRAM de ~8GB a ~2GB) | Permite correr LLMs de alta calidad en GPU gratuita T4 | Autonomía técnica, soberanía de datos y costo cero |

---


### ✅ Checklist de Verificación para la Entrega (TP1 - Parte 1)
Antes de guardar y descargar tu cuaderno, verifica los siguientes puntos:
* [ ] El entorno de ejecución está configurado con **GPU T4**.
* [ ] Las celdas del **Bloque 0** se ejecutaron exitosamente y el modelo quedó montado en VRAM.
* [ ] Se realizaron las pruebas interactivas en los **Experimentos 1, 2, 3 y 4** utilizando los formularios interactivos.
* [ ] Se redactaron las respuestas y reflexiones de negocio en los cuatro recuadros `✍️ Tu análisis aquí:`.
* [ ] **Guardado final:** Haz clic en `Archivo` -> `Guardar una copia en Drive` o `Descargar .ipynb` para respaldar tu trabajo. Este archivo se sube al aula virtual.

---

### 🚀 Próximo Paso: Clase 2
En la próxima sesión abordaremos **Prompt Engineering Avanzado y Estructuración de Datos**, donde utilizaremos este mismo motor local para forzar salidas estrictas en formato **JSON y esquemas estructurados**, base indispensable para la integración de LLMs en sistemas de gestión corporativos.